# Experimento principal en español — IMAC

Preparado el **9 de noviembre de 2026**, después del piloto español de cinco ajustes.
Este archivo es nuevo y autónomo; conserva aparte el piloto y sus salidas.

**Diseño:** GPT-2 pequeño adaptado al español, Wikipedia `20231101.es`, 512 ejemplos,
contextos de 64 tokens y objetivos de 128; 0/50/100 % de objetivos sintéticos, cinco ciclos,
tres repeticiones. Son **48 ajustes**: tres M0 y 45 posteriores. M0 se comparte dentro
de cada repetición. Todos los ajustes vuelven al modelo preentrenado y optimizador nuevo.

**Preparación no equivale a ejecución:** se verificaron controles offline y sintaxis.
Este flujo completo y sus checkpoints en Drive aún no se probaron en Colab. La primera
ejecución queda limitada a **una etapa** para verificar persistencia y reanudación.
`RUN_EXPERIMENT=False` evita iniciar ajustes al usar «Ejecutar todas» por primera vez.

## Orden de uso

1. Importar este notebook, elegir GPU y utilizar una sesión limpia. Instalar dependencias.
2. Ejecutar controles y definiciones. La prueba diminuta verifica guardado/restauración,
   sin descargar el modelo grande ni ejecutar el estudio.
3. Revisar la ruta de salida. Activar `PREPARE_DRIVE=True` y completar personalmente la
   autorización de montaje si se desea guardar en Drive. El programa solo usa la carpeta
   de este experimento; montar Drive da acceso al entorno y requiere tu consentimiento.
4. Activar `RUN_EXPERIMENT=True` con `MAX_NEW_STAGES=1` para la primera etapa. Comprobar
   sus archivos y repetir esa celda para verificar que se omite la etapa confirmada.
5. Para continuar el resto, usar `MAX_NEW_STAGES=None` con la misma configuración y carpeta.
   Ejecutar una sola sesión a la vez. No cambiar parámetros mirando métricas de prueba.

Se estimaron unas 2.67 h de cálculo por extrapolación del piloto, **sin validar la duración
principal ni incluir el costo adicional de checkpoints, Drive e interrupciones**. Dejar
margen para análisis y redacción. No se compra cómputo ni se requieren claves de API.


## 1. Dependencias

Se conservan las versiones observadas en el piloto para los paquetes principales.
PyTorch proviene del runtime de Colab: no se reemplaza automáticamente. Al reanudar se
exigen las mismas versiones esenciales, Python, CUDA y nombre de GPU que en la primera
sesión. Una incompatibilidad detiene el programa, no mezcla ejecuciones silenciosamente.


In [ ]:
import sys
import subprocess
import importlib.util
if sys.version_info < (3, 11) or importlib.util.find_spec('torch') is None:
    raise RuntimeError('Usar un runtime reciente de Colab con PyTorch y GPU.')
if any(name in sys.modules for name in ('transformers', 'datasets')):
    raise RuntimeError('Reiniciar sesión antes de reinstalar paquetes ya importados.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
    'transformers==5.22.2', 'datasets==5.1.0', 'safetensors==0.8.0',
    'matplotlib==3.11.2', 'numpy==2.4.6', 'tokenizers==0.23.3'], check=True)
print('Instalación terminada. Si Colab pide reiniciar, hacerlo antes de continuar con imports.')


## 2. Controles del protocolo y de reanudación

Se conservan Unicode NFC, tildes y ñ. Las particiones del piloto **no se cambian**:
hash de semilla 20261109 e ID de artículo, 80/10/10 aproximado. Ahora se preparan 512
ejemplos de entrenamiento, 128 de validación y 128 de prueba de artículos disjuntos.
Se usa un prefijo acotado de 2000 artículos, una ventana por artículo y controles de
duplicados. Si faltan candidatos, se detiene antes de ajustar.

La validación fija W (hasta 32 palabras); prueba no elige parámetros ni checkpoints.
Wikipedia es referencia original, sin certificar autoría humana individual. El modelo
ya se adaptó a Wikipedia: no se acredita ausencia de solapamiento con preentrenamiento.


In [ ]:
"""Controles del protocolo; solo biblioteca estándar, también embebidos en el notebook."""
import hashlib
import json
import math
import random
import re
from collections import Counter


def stable_seed(*parts):
    raw = json.dumps(parts, ensure_ascii=False, separators=(',', ':')).encode()
    return int.from_bytes(hashlib.sha256(raw).digest()[:4], 'big')


def articles(rows):
    """WikiText: solo un encabezado = título = inicia otro artículo.

    Se retiran encabezados de cualquier nivel; se conservan párrafos y saltos.
    Un fragmento inicial sin título se conserva como artículo propio.
    """
    out, body, title = [], [], 'untitled'
    for row in rows:
        text = row.strip()
        if re.match(r'^=\s+[^=].*?\s+=$', text):
            if body:
                out.append({'article': f'{len(out)}:{title}', 'text': '\n'.join(body)})
            body, title = [], text.strip('= ').strip()
        elif text and not re.match(r'^=+\s.*?\s=+$', text):
            body.append(text)
    if body:
        out.append({'article': f'{len(out)}:{title}', 'text': '\n'.join(body)})
    return out


def windows(docs, context_len, target_len, split):
    width = context_len + target_len
    out = []
    for doc in docs:
        for offset in range(0, len(doc['tokens']) - width + 1, width):
            block = doc['tokens'][offset:offset + width]
            out.append({'id': f"{split}:{doc['article']}:{offset}",
                        'split': split, 'article': doc['article'], 'offset': offset,
                        'context': block[:context_len], 'target': block[context_len:]})
    return out


def labelled(context, target, pad_id, length=None):
    if not context or not target:
        raise ValueError('Contexto y objetivo deben ser no vacíos.')
    ids = list(context) + list(target)
    length = len(ids) if length is None else length
    if length < len(ids):
        raise ValueError('El relleno no puede truncar el objetivo.')
    padding = length - len(ids)
    return (ids + [pad_id] * padding,
            [1] * len(ids) + [0] * padding,
            [-100] * len(context) + list(target) + [-100] * padding)


def mix_targets(human, synthetic, alpha, seed):
    """Reemplaza ejemplos completos; longitud fija implica proporción exacta de tokens."""
    if not human or not 0 <= alpha <= 1:
        raise ValueError('Corpus vacío o proporción inválida.')
    count = len(human) * alpha
    if not math.isclose(count, round(count)):
        raise ValueError('La proporción debe dar un número entero de ejemplos.')
    length = len(human[0]['target'])
    if any(len(x['target']) != length for x in human):
        raise ValueError('Objetivos humanos de longitud desigual.')
    if alpha and (synthetic is None or len(synthetic) != len(human)
                  or any(len(x) != length for x in synthetic)):
        raise ValueError('Objetivos sintéticos incompletos o de longitud desigual.')
    chosen = set(random.Random(seed).sample(range(len(human)), round(count)))
    return [dict(x, context=list(x['context']),
                 target=list(synthetic[i] if i in chosen else x['target']),
                 synthetic=i in chosen) for i, x in enumerate(human)]


WORD_PATTERN = r"\w+(?:['’]\w+)*"


def words(text):
    return re.findall(WORD_PATTERN, text.lower(), flags=re.UNICODE)


def lexical_metrics(texts, limit):
    if not texts or limit < 4:
        raise ValueError('Se requieren textos y al menos cuatro palabras por texto.')
    seqs = [words(t) for t in texts]
    short = sum(len(s) < limit for s in seqs)
    result = {'word_limit': limit, 'n_texts': len(texts), 'short_texts': short,
              'distinct_2': None, 'distinct_3': None, 'repeat_4_mean': None}
    if short:
        return result
    seqs = [s[:limit] for s in seqs]
    for n in (2, 3):
        grams = [tuple(s[i:i+n]) for s in seqs for i in range(len(s)-n+1)]
        result[f'distinct_{n}'] = len(set(grams)) / len(grams)
    repeats = []
    for s in seqs:
        grams = [tuple(s[i:i+4]) for i in range(len(s)-3)]
        repeats.append(1 - len(set(grams)) / len(grams))
    result['repeat_4_mean'] = sum(repeats) / len(repeats)
    return result


"""Selección de artículos del corpus español; sin librerías externas."""
import hashlib
import random
import unicodedata
base_words = words
base_lexical_metrics = lexical_metrics


def normalized_text(text):
    return unicodedata.normalize('NFC', text.replace('\r\n', '\n').replace('\r', '\n')).strip()


def words(text):
    return base_words(normalized_text(text))


def lexical_metrics(texts, limit):
    return base_lexical_metrics([normalized_text(t) for t in texts], limit)


def article_partition(article_id, seed):
    digest = hashlib.sha256(f'{seed}:article_split:{article_id}'.encode()).digest()
    bucket = int.from_bytes(digest[:8], 'big') % 100
    return 'train' if bucket < 80 else 'validation' if bucket < 90 else 'test'


def spanish_candidates(records, tokenize, context_len, target_len, seed, max_articles):
    """Prefijo acotado del streaming; una ventana aleatoria por artículo elegible.

    Las particiones son propias (80/10/10 aproximado por hash), no oficiales.
    Los artículos asignados a test no se tokenizan ni se usan para medir.
    El prefijo no es una muestra representativa de toda Wikipedia.
    """
    if context_len <= 0 or target_len <= 0 or max_articles <= 0:
        raise ValueError('Longitudes y límite de artículos deben ser positivos.')
    pools = {'train': [], 'validation': []}
    width = context_len + target_len
    audit = {'scanned_articles': 0, 'reserved_test_articles': 0,
             'too_short_articles': 0, 'duplicate_ids': 0}
    seen = set()
    for row in records:
        if audit['scanned_articles'] >= max_articles:
            break
        audit['scanned_articles'] += 1
        article = str(row['id'])
        if article in seen:
            audit['duplicate_ids'] += 1
            continue
        seen.add(article)
        split = article_partition(article, seed)
        if split == 'test':
            audit['reserved_test_articles'] += 1
            continue
        text = normalized_text(row['text'])
        tokens = tokenize(text)
        count = len(tokens) // width
        if not count:
            audit['too_short_articles'] += 1
            continue
        offset = random.Random(stable_seed(seed, 'window', article)).randrange(count) * width
        block = tokens[offset:offset + width]
        pools[split].append({'id': f'es:{split}:{article}:{offset}', 'split': split,
            'article': article, 'title': row.get('title', ''), 'url': row.get('url', ''),
            'offset': offset, 'text_sha256': hashlib.sha256(text.encode()).hexdigest(),
            'context': block[:context_len], 'target': block[context_len:]})
    return pools, audit


def select_spanish_rows(pools, train_count, valid_count, seed):
    selected = {}
    for split, count in (('train', train_count), ('validation', valid_count)):
        if len(pools[split]) < count:
            raise ValueError(f'{split}: solo {len(pools[split])} artículos elegibles; se necesitan {count}. '
                             'Revisar el límite antes de generar o entrenar.')
        selected[split] = random.Random(stable_seed(seed, 'data', split)).sample(pools[split], count)
    train_ids = {r['article'] for r in selected['train']}
    train_text = {r['text_sha256'] for r in selected['train']}
    if any(r['article'] in train_ids or r['text_sha256'] in train_text for r in selected['validation']):
        raise ValueError('Artículo o contenido idéntico entre entrenamiento y validación.')
    for split, rows in selected.items():
        if len({r['text_sha256'] for r in rows}) != len(rows):
            raise ValueError(f'Contenido duplicado en la muestra de {split}; revisar sin reemplazos silenciosos.')
    return selected


"""Protocolo y persistencia del estudio principal; solo biblioteca estándar."""
import hashlib
import json
import random
from pathlib import Path


def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(',', ':'), allow_nan=False).encode()).hexdigest()


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_name(path.name+'.pending')
    pending.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    pending.replace(path)


def make_plan(repetitions=3, cycles=5, alphas=(0.0, 0.5, 1.0)):
    jobs = []
    for rep in range(repetitions):
        initial = f'r{rep}_M0'
        jobs.append({'id': initial, 'rep': rep, 'cycle': 0, 'alpha': None, 'parent': None})
        for alpha in alphas:
            for cycle in range(1, cycles+1):
                name = f'r{rep}_a{round(alpha*100):03d}_g{cycle}'
                parent = initial if cycle == 1 else f'r{rep}_a{round(alpha*100):03d}_g{cycle-1}'
                jobs.append({'id': name, 'rep': rep, 'cycle': cycle, 'alpha': alpha, 'parent': parent})
    return jobs


def freeze_protocol(root, spec, environment):
    spec = json.loads(json.dumps(spec, ensure_ascii=False, allow_nan=False))
    environment = json.loads(json.dumps(environment, ensure_ascii=False, allow_nan=False))
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    path = root/'protocol.json'
    record = {'spec': spec, 'environment': environment,
              'signature': canonical_hash({'spec': spec, 'environment': environment})}
    if path.exists():
        previous = json.loads(path.read_text(encoding='utf-8'))
        if previous != record:
            raise ValueError('El protocolo o entorno cambió. No mezclar ejecuciones; conservar esta carpeta.')
    else:
        if any(root.iterdir()):
            raise ValueError('Directorio no vacío sin protocolo identificable. Usar otra carpeta.')
        atomic_json(path, record)
    return record['signature']


def commit_stage(folder, signature, artifacts, details):
    folder = Path(folder)
    for name in artifacts:
        if Path(name).is_absolute() or '..' in Path(name).parts:
            raise ValueError('Artefacto fuera del directorio de etapa.')
    record = {'signature': signature, 'files': {name: file_hash(folder/name) for name in artifacts},
              'details': details}
    atomic_json(folder/'complete.json', record)


def stage_complete(folder, signature):
    folder = Path(folder)
    marker = folder/'complete.json'
    if not marker.exists():
        return False
    record = json.loads(marker.read_text(encoding='utf-8'))
    if record['signature'] != signature:
        raise ValueError('Marcador de otro protocolo.')
    for name, digest in record['files'].items():
        if Path(name).is_absolute() or '..' in Path(name).parts:
            raise ValueError('Ruta de artefacto inválida.')
        path = folder/name
        if not path.is_file() or file_hash(path) != digest:
            raise ValueError(f'Artefacto ausente o modificado: {path}. No se sobrescribe silenciosamente.')
    return True


def final_selection(pools, counts, seed):
    selected, article_ids, contents, windows = {}, set(), set(), set()
    for split, count in zip(('train', 'validation', 'test'), counts):
        if len(pools[split]) < count:
            raise ValueError(f'{split}: faltan artículos elegibles ({len(pools[split])}/{count}).')
        selected[split] = random.Random(stable_seed(seed, 'final_selection', split)).sample(pools[split], count)
        for row in selected[split]:
            block = tuple(row['context']+row['target'])
            if row['article'] in article_ids or row['text_sha256'] in contents or block in windows:
                raise ValueError('Duplicados en la muestra final; no se sustituyen según resultados.')
            article_ids.add(row['article'])
            contents.add(row['text_sha256'])
            windows.add(block)
    return selected


def cached_chunk(path, input_hash):
    path = Path(path)
    if not path.exists():
        return None
    record = json.loads(path.read_text(encoding='utf-8'))
    if record['input_hash'] != input_hash or record['output_hash'] != canonical_hash(record['outputs']):
        raise ValueError('Lote guardado incompatible o modificado.')
    return record['outputs']


def save_chunk(path, input_hash, outputs):
    atomic_json(path, {'input_hash': input_hash, 'output_hash': canonical_hash(outputs), 'outputs': outputs})


In [ ]:
"""Controles offline: ejecutar con python -m unittest discover -s Experimentos -v."""
import unittest


class ProtocolTests(unittest.TestCase):
    def test_article_boundaries_and_subheadings(self):
        rows = [' = One = \n', 'alpha', ' == Sub == \n', 'beta', ' = Two = \n', 'gamma']
        docs = articles(rows)
        self.assertEqual(len(docs), 2)
        self.assertIn('beta', docs[0]['text'])
        self.assertNotIn('gamma', docs[0]['text'])
        self.assertNotIn('One', docs[0]['text'])

    def test_no_cross_article_or_overlapping_windows(self):
        docs = [{'article': 'a', 'tokens': list(range(13))},
                {'article': 'b', 'tokens': list(range(100, 105))}]
        blocks = windows(docs, 2, 3, 'train')
        self.assertEqual([x['offset'] for x in blocks], [0, 5, 0])
        self.assertEqual(blocks[1]['context'] + blocks[1]['target'], list(range(5, 10)))
        self.assertEqual(blocks[2]['target'], [102, 103, 104])
        self.assertEqual(len({x['id'] for x in blocks}), 3)

    def test_context_and_padding_are_not_training_targets(self):
        ids, mask, labels = labelled([8, 9], [10, 11, 12], pad_id=1, length=7)
        self.assertEqual(ids, [8, 9, 10, 11, 12, 1, 1])
        self.assertEqual(mask, [1, 1, 1, 1, 1, 0, 0])
        self.assertEqual(labels, [-100, -100, 10, 11, 12, -100, -100])

    def test_mixture_is_exact_and_keeps_human_context(self):
        human = [{'id': str(i), 'context': [i], 'target': [7, 7, 7, 7]} for i in range(4)]
        synthetic = [[9, 9, 9, 9] for _ in human]
        out = mix_targets(human, synthetic, .5, 42)
        self.assertEqual(sum(x['synthetic'] for x in out), 2)
        self.assertEqual(sum(x['target'].count(9) for x in out), 8)
        self.assertEqual([x['context'] for x in out], [[i] for i in range(4)])
        self.assertEqual(human[0]['target'], [7, 7, 7, 7])
        with self.assertRaises(ValueError):
            mix_targets(human, [[9]] * 4, 1, 42)
        with self.assertRaises(ValueError):
            mix_targets(human, synthetic, .3, 42)

    def test_seeds_stable_and_namespaces_separate(self):
        self.assertEqual(stable_seed(42, 'train', 1), stable_seed(42, 'train', 1))
        self.assertNotEqual(stable_seed(42, 'train', 1), stable_seed(42, 'eval', 1))

    def test_lexical_metrics_do_not_cross_text_boundaries(self):
        m = lexical_metrics(['A b a b a b', 'c d c d c d'], 6)
        self.assertAlmostEqual(m['distinct_2'], 4 / 10)
        self.assertAlmostEqual(m['repeat_4_mean'], 1 / 3)
        self.assertEqual(m['short_texts'], 0)

    def test_short_texts_are_not_silently_dropped(self):
        m = lexical_metrics(['a b c d', 'a'], 4)
        self.assertEqual(m['short_texts'], 1)
        self.assertIsNone(m['distinct_2'])
        self.assertIsNone(m['repeat_4_mean'])



"""Pruebas adicionales del piloto español, sin GPU ni descargas."""
import unittest


class SpanishProtocolTests(unittest.TestCase):
    def test_accents_and_enye_are_preserved(self):
        self.assertEqual(normalized_text('  cancio\u0301n\r\nniño  '), 'canción\nniño')
        self.assertEqual(words(normalized_text('Árbol, niño y acción.')), ['árbol', 'niño', 'y', 'acción'])
        self.assertEqual(words('cancio\u0301n y niño'), ['canción', 'y', 'niño'])
        self.assertEqual(lexical_metrics(['cancio\u0301n y niño feliz'], 4),
                         lexical_metrics(['canción y niño feliz'], 4))

    def test_partition_is_stable_and_uses_all_three_folds(self):
        folds = [article_partition(str(i), 42) for i in range(200)]
        self.assertEqual(set(folds), {'train', 'validation', 'test'})
        self.assertEqual(article_partition('15', 42), article_partition('15', 42))

    def test_article_stays_in_one_partition_and_only_one_window(self):
        rows = [{'id': str(i), 'title': f'Título {i}', 'url': 'https://es.wikipedia.org/',
                 'text': 'Texto en español con suficientes palabras.'} for i in range(70)]
        seen_texts = []
        def tokenize(text):
            seen_texts.append(text)
            return list(range(20))
        pools, audit = spanish_candidates(rows, tokenize, 2, 3, seed=42, max_articles=60)
        self.assertEqual(audit['scanned_articles'], 60)
        self.assertLess(len(seen_texts), 60)  # Los artículos de test no se tokenizan.
        all_rows = pools['train'] + pools['validation']
        self.assertEqual(len(all_rows), len({r['article'] for r in all_rows}))
        self.assertFalse(set(r['article'] for r in pools['train']) &
                         set(r['article'] for r in pools['validation']))
        for row in all_rows:
            self.assertEqual(len(row['context']), 2)
            self.assertEqual(len(row['target']), 3)
            self.assertEqual(row['split'], article_partition(row['article'], 42))
            self.assertEqual(row['offset'] % 5, 0)

    def test_short_articles_are_reported_without_padding(self):
        rows = [{'id': str(i), 'title': 'Corto', 'text': 'corto'} for i in range(40)]
        pools, audit = spanish_candidates(rows, lambda t: [1, 2], 2, 3, 42, 40)
        self.assertEqual(pools, {'train': [], 'validation': []})
        self.assertGreater(audit['too_short_articles'], 0)

    def test_duplicate_reference_text_across_folds_is_rejected(self):
        pools = {'train': [{'article': 'a', 'text_sha256': 'same'}],
                 'validation': [{'article': 'b', 'text_sha256': 'same'}]}
        with self.assertRaises(ValueError):
            select_spanish_rows(pools, 1, 1, 42)

    def test_sample_counts_are_exact_and_pool_must_suffice(self):
        pools = {s: [{'article': f'{s}{i}', 'text_sha256': f'{s}{i}'} for i in range(10)]
                 for s in ('train', 'validation')}
        selected = select_spanish_rows(pools, 4, 3, 42)
        self.assertEqual(len(selected['train']), 4)
        self.assertEqual(len(selected['validation']), 3)
        self.assertEqual(selected, select_spanish_rows(pools, 4, 3, 42))
        with self.assertRaises(ValueError):
            select_spanish_rows(pools, 11, 3, 42)



import json
import tempfile
import unittest
from pathlib import Path


class MainProtocolTests(unittest.TestCase):
    def test_48_adjustments_and_shared_initial_points(self):
        jobs = make_plan()
        self.assertEqual(len(jobs), 48)
        self.assertEqual(sum(x['cycle'] == 0 for x in jobs), 3)
        for job in jobs:
            if job['cycle'] == 1:
                self.assertEqual(job['parent'], f"r{job['rep']}_M0")
            elif job['cycle'] > 1:
                self.assertIn(f"g{job['cycle']-1}", job['parent'])

    def test_resume_rejects_configuration_or_environment_change(self):
        with tempfile.TemporaryDirectory(dir=Path.cwd()) as temp:
            root = Path(temp)
            signature = freeze_protocol(root, {'seed': 1}, {'torch': 'x'})
            self.assertEqual(signature, freeze_protocol(root, {'seed': 1}, {'torch': 'x'}))
            with self.assertRaises(ValueError):
                freeze_protocol(root, {'seed': 2}, {'torch': 'x'})
            with self.assertRaises(ValueError):
                freeze_protocol(root, {'seed': 1}, {'torch': 'y'})

    def test_json_roundtrip_does_not_change_tuple_configuration(self):
        with tempfile.TemporaryDirectory(dir=Path.cwd()) as temp:
            spec = {'alphas': (0.0, 0.5, 1.0)}
            first = freeze_protocol(Path(temp), spec, {'torch': 'x'})
            self.assertEqual(first, freeze_protocol(Path(temp), spec, {'torch': 'x'}))

    def test_incomplete_stage_is_not_skipped_and_corruption_stops_resume(self):
        with tempfile.TemporaryDirectory(dir=Path.cwd()) as temp:
            root = Path(temp)
            (root/'metrics.json').write_text('{}', encoding='utf-8')
            self.assertFalse(stage_complete(root, 'spec'))
            commit_stage(root, 'spec', ['metrics.json'], {'rep': 0})
            self.assertTrue(stage_complete(root, 'spec'))
            (root/'metrics.json').write_text('{"changed":true}', encoding='utf-8')
            with self.assertRaises(ValueError):
                stage_complete(root, 'spec')

    def test_validation_and_test_stay_separate(self):
        pools = {s: [{'id': f'{s}{i}', 'article': f'{s}{i}', 'text_sha256': f'{s}{i}',
                       'context': [100*j+i], 'target': [100*j+i+2]} for i in range(6)]
                 for j, s in enumerate(('train', 'validation', 'test'))}
        selected = final_selection(pools, (4, 2, 3), 19)
        self.assertEqual([len(selected[s]) for s in pools], [4, 2, 3])
        pools['test'] = []
        with self.assertRaises(ValueError):
            final_selection(pools, (4, 2, 3), 19)

    def test_cross_partition_content_duplicates_rejected(self):
        pools = {s: [{'id': s, 'article': s, 'text_sha256': 'identical',
                       'context': [1], 'target': [2]}] for s in ('train', 'validation', 'test')}
        with self.assertRaises(ValueError):
            final_selection(pools, (1, 1, 1), 9)

    def test_completed_generation_chunk_is_reused_and_detects_corruption(self):
        with tempfile.TemporaryDirectory(dir=Path.cwd()) as temp:
            path = Path(temp)/'batch.json'
            self.assertIsNone(cached_chunk(path, 'request'))
            save_chunk(path, 'request', [{'id': 'a', 'target_ids': [9]}])
            self.assertEqual(cached_chunk(path, 'request')[0]['target_ids'], [9])
            with self.assertRaises(ValueError):
                cached_chunk(path, 'different-request')
            value = json.loads(path.read_text(encoding='utf-8'))
            value['outputs'][0]['target_ids'] = [10]
            path.write_text(json.dumps(value), encoding='utf-8')
            with self.assertRaises(ValueError):
                cached_chunk(path, 'request')



suite = unittest.TestSuite([
    unittest.defaultTestLoader.loadTestsFromTestCase(ProtocolTests),
    unittest.defaultTestLoader.loadTestsFromTestCase(SpanishProtocolTests),
    unittest.defaultTestLoader.loadTestsFromTestCase(MainProtocolTests),
])
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful(), 'No continuar si falla algún control.'


## 3. Funciones de aprendizaje y métricas usadas en el piloto

Se reutiliza el ajuste completo de una época, AdamW 2e-5 (betas .9/.999, epsilon 1e-8,
decay .01 uniforme), clipping 1, lote efectivo 8 y microbatch 2. Son 64 actualizaciones
y 65 536 tokens objetivo por ajuste. Contexto/relleno se enmascaran con `-100`.

Muestreo T=1, top-p=.95, sin top-k ni penalización de repetición; exactamente 128 tokens,
excluyendo IDs especiales, incluido EOS. Un RNG por solicitud. Las semillas de evaluación
se emparejan entre etapas de la misma repetición; las de entrenamiento/datos son distintas.

Distinct-2/3 por corpus sin cruzar textos, repetición media por texto, dispersión media
de coseno con MiniLM multilingüe y PPL de continuaciones originales reservadas. La métrica
semántica utiliza mean pooling/L2, máximo 128 wordpieces y truncamientos registrados.
Las etapas con salidas más cortas que W conservan esos textos y dejan ausentes las métricas
léxicas/semánticas afectadas. Se guardan muestras sin filtrar por idioma/calidad.


In [ ]:
import csv
import importlib.metadata
import os
import platform
import shutil
import sys
import time
import uuid
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

# No se necesitan tokens para estos repositorios públicos.
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, GPT2Config, GPT2LMHeadModel
import matplotlib.pyplot as plt


@dataclass(frozen=True)
class Config:
    language: str = 'es'
    model: str = 'datificate/gpt2-small-spanish'
    dataset: str = 'wikimedia/wikipedia'
    subset: str = '20231101.es'
    embedding: str = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
    article_pool: int = 2000
    seed: int = 20261109
    train_examples: int = 64
    eval_examples: int = 32
    context_tokens: int = 64
    target_tokens: int = 128
    cycles: int = 2
    alphas: tuple = (0.0, 1.0)
    micro_batch: int = 2
    effective_batch: int = 8
    sample_batch: int = 8
    learning_rate: float = 2e-5
    betas: tuple = (0.9, 0.999)
    epsilon: float = 1e-8
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    temperature: float = 1.0
    top_p: float = 0.95
    max_words: int = 32
    embedding_max_tokens: int = 128
    allow_cpu: bool = False
    save_checkpoints: bool = False


CFG = Config()


def validate_config(cfg):
    if cfg.language != 'es' or cfg.subset != '20231101.es':
        raise ValueError('Este piloto debe trabajar con el corpus en español.')
    if (cfg.train_examples != 64 or cfg.eval_examples != 32 or cfg.cycles != 2
            or tuple(cfg.alphas) != (0.0, 1.0)):
        raise ValueError('Este notebook está acotado al piloto; no activar el estudio final aquí.')
    if cfg.train_examples % cfg.effective_batch or cfg.effective_batch % cfg.micro_batch:
        raise ValueError('Los lotes deben dividir el corpus y el lote efectivo exactamente.')
    if not 0 < cfg.top_p <= 1 or cfg.temperature <= 0:
        raise ValueError('Parámetros de muestreo inválidos.')


def seed_training(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def weight_hash(model):
    digest = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        digest.update(name.encode())
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def reset_for_fit(model, base_state, expected_hash, cfg, seed):
    model.load_state_dict(base_state, strict=True)
    model.zero_grad(set_to_none=True)
    observed_hash = weight_hash(model)
    if observed_hash != expected_hash:
        raise RuntimeError('La inicialización no coincide con el modelo base.')
    seed_training(seed)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate, betas=cfg.betas,
        eps=cfg.epsilon, weight_decay=cfg.weight_decay, foreach=False)
    if len(optimizer.state):
        raise RuntimeError('El optimizador no empieza vacío.')
    return optimizer, observed_hash


def batch_tensors(rows, pad_id, device):
    packed = [labelled(x['context'], x['target'], pad_id) for x in rows]
    return {name: torch.tensor([x[i] for x in packed], dtype=torch.long, device=device)
            for i, name in enumerate(('input_ids', 'attention_mask', 'labels'))}


def fit_one(model, base_state, base_hash, rows, cfg, seed, device, pad_id):
    optimizer, initial_hash = reset_for_fit(model, base_state, base_hash, cfg, seed)
    generator = torch.Generator().manual_seed(stable_seed(seed, 'shuffle'))
    loader = DataLoader(rows, batch_size=cfg.micro_batch, shuffle=True,
                        generator=generator, collate_fn=list, num_workers=0)
    accumulation = cfg.effective_batch // cfg.micro_batch
    model.train()
    losses, updates, seen = [], 0, 0
    for step, batch in enumerate(loader, start=1):
        data = batch_tensors(batch, pad_id, device)
        labels = data['labels']
        assert (labels[:, :cfg.context_tokens] == -100).all().item()
        assert (labels[:, cfg.context_tokens:] != -100).sum().item() == len(batch) * cfg.target_tokens
        output = model(**data, use_cache=False)
        if not torch.isfinite(output.loss).item():
            raise RuntimeError('Pérdida no finita; no se cambian parámetros automáticamente.')
        losses.append(float(output.loss.detach().cpu()))
        (output.loss / accumulation).backward()
        seen += len(batch) * cfg.target_tokens
        if step % accumulation == 0:
            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            if not torch.isfinite(norm).item():
                raise RuntimeError('Gradiente no finito.')
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            updates += 1
    assert updates == len(rows) // cfg.effective_batch
    assert seen == len(rows) * cfg.target_tokens
    del optimizer
    model.eval()
    return {'initial_weight_sha256': initial_hash, 'train_seed': seed,
            'optimizer_initial_entries': 0, 'updates': updates,
            'target_tokens_seen': seen, 'train_loss_mean': sum(losses)/len(losses),
            'final_weight_sha256': weight_hash(model)}


In [ ]:
def nucleus_draw(logits, generators, cfg, forbidden_ids):
    logits = logits.float().clone() / cfg.temperature
    logits[:, forbidden_ids] = -float('inf')
    values, indices = torch.sort(logits, descending=True, dim=-1)
    probs = torch.softmax(values, dim=-1)
    remove = (torch.cumsum(probs, dim=-1) - probs) >= cfg.top_p
    values[remove] = -float('inf')
    probs = torch.softmax(values, dim=-1)
    draws = [torch.multinomial(p, 1, generator=g) for p, g in zip(probs, generators)]
    return indices.gather(1, torch.stack(draws))


@torch.inference_mode()
def sample_targets(model, rows, cfg, device, forbidden_ids, namespace, progress=True):
    """Un RNG por solicitud; no se hereda el RNG de entrenamiento.

    Política piloto: longitud fija y ningún token especial (incluye EOS/PAD/BOS).
    Se conserva el token que cruza el umbral top-p. Sin top-k ni penalizaciones.
    """
    model.eval()
    targets, seeds = [], []
    for start in range(0, len(rows), cfg.sample_batch):
        batch = rows[start:start+cfg.sample_batch]
        batch_seeds = [stable_seed(cfg.seed, 'sampling', namespace, x['id']) for x in batch]
        generators = [torch.Generator(device=device).manual_seed(s) for s in batch_seeds]
        ids = torch.tensor([x['context'] for x in batch], dtype=torch.long, device=device)
        generated, cache = [], None
        attention = torch.ones_like(ids)
        for _ in range(cfg.target_tokens):
            output = model(input_ids=ids, attention_mask=attention,
                           past_key_values=cache, use_cache=True)
            ids = nucleus_draw(output.logits[:, -1, :], generators, cfg, forbidden_ids)
            generated.append(ids)
            cache = output.past_key_values
            attention = torch.cat([attention, torch.ones_like(ids)], dim=1)
        result = torch.cat(generated, dim=1).cpu().tolist()
        assert all(len(x) == cfg.target_tokens for x in result)
        assert all(not (set(x) & set(forbidden_ids)) for x in result)
        targets.extend(result)
        seeds.extend(batch_seeds)
        if progress:
            print(f'  Muestreo {min(start+cfg.sample_batch, len(rows))}/{len(rows)}', flush=True)
    return targets, seeds


def torch_smoke_checks():
    """Sin descargas: GPT-2 diminuto; pérdida causal, caché, semillas y reinicio."""
    from dataclasses import replace
    seed_training(17)
    toy = GPT2LMHeadModel(GPT2Config(vocab_size=32, n_embd=16, n_inner=32,
        n_layer=1, n_head=2, n_positions=32,
        resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0,
        pad_token_id=1, bos_token_id=2, eos_token_id=2)).cpu()
    config = replace(CFG, context_tokens=2, target_tokens=4,
                     micro_batch=1, effective_batch=2, sample_batch=2)
    rows = [{'id': str(i), 'context': [4, 5], 'target': [6, 7, 8, 9]} for i in range(2)]
    data = batch_tensors(rows, 1, 'cpu')
    toy.eval()
    output = toy(**data, use_cache=False)
    manual = F.cross_entropy(output.logits[:, :-1, :].reshape(-1, 32),
                             data['labels'][:, 1:].reshape(-1), ignore_index=-100)
    assert torch.allclose(output.loss, manual, atol=1e-6)
    assert data['labels'][:, 1:].ne(-100).sum().item() == 8
    a, _ = sample_targets(toy, rows, config, 'cpu', [0, 1, 2, 3], 'check', False)
    torch.rand(123)  # Perturbar RNG global no debe cambiar la generación.
    b, _ = sample_targets(toy, rows, config, 'cpu', [0, 1, 2, 3], 'check', False)
    assert a == b
    base, signature = cpu_state(toy), weight_hash(toy)
    fit_one(toy, base, signature, rows, config, 99, 'cpu', 1)
    assert weight_hash(toy) != signature
    optimizer, observed = reset_for_fit(toy, base, signature, config, 99)
    assert observed == signature and len(optimizer.state) == 0
    # El relleno debe estar excluido también después del desplazamiento causal.
    ids, mask, labels = labelled([4, 5], [6, 7], 1, 7)
    padded = {k: torch.tensor([v]) for k, v in
              zip(('input_ids', 'attention_mask', 'labels'), (ids, mask, labels))}
    toy.eval()
    output = toy(**padded, use_cache=False)
    loss = F.cross_entropy(output.logits[:, :-1].reshape(-1, 32),
                           padded['labels'][:, 1:].reshape(-1), ignore_index=-100)
    assert torch.allclose(output.loss, loss, atol=1e-6)
    del optimizer, toy
    print('Controles PyTorch: pérdida causal, relleno, muestreo y reinicio correctos.')


In [ ]:
def write_json(path, obj):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temp.replace(path)


def write_jsonl(path, rows):
    with Path(path).open('w', encoding='utf-8') as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')


def hash_json(obj):
    return hashlib.sha256(json.dumps(obj, ensure_ascii=False, sort_keys=True).encode()).hexdigest()


class Clock:
    def __init__(self, path, device):
        self.path, self.device, self.records = path, device, []

    @contextmanager
    def measure(self, stage, **tags):
        if self.device == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        status = 'error'
        try:
            yield
            status = 'ok'
        finally:
            if self.device == 'cuda':
                torch.cuda.synchronize()
            self.records.append(dict(stage=stage, seconds=time.perf_counter()-start,
                                     status=status, **tags))
            write_json(self.path, self.records)


def select_data(cfg, tokenizer, revision):
    # El repositorio solo tiene train: nuestras particiones se definen por artículo.
    # Streaming evita descargar toda la Wikipedia; puede transferir buffers adicionales.
    raw = load_dataset(cfg.dataset, cfg.subset, split='train', revision=revision,
                       streaming=True, token=False)
    def tokenize(text):
        return tokenizer(text, add_special_tokens=False, truncation=False,
                         verbose=False)['input_ids']
    pools, audit = spanish_candidates(raw, tokenize, cfg.context_tokens, cfg.target_tokens,
                                      cfg.seed, cfg.article_pool)
    print('Artículos examinados:', audit['scanned_articles'],
          '| Candidatos:', {k: len(v) for k, v in pools.items()}, flush=True)
    selected = select_spanish_rows(pools, cfg.train_examples, cfg.eval_examples, cfg.seed)
    fingerprints = {'selection': audit, 'official_split': 'train',
        'partition_policy': 'SHA256(seed:article_split:id) mod100; train<80, validation<90, test>=90',
        'sampling_scope': f'Prefijo de hasta {cfg.article_pool} artículos; una ventana por artículo',
        'test_tokenized_or_evaluated': False,
        'normalization': 'NFC y saltos de línea; se conservan tildes y ñ',
        'provenance': 'Referencia Wikipedia; no se acredita autoría humana individual'}
    for split, chosen in selected.items():
        fingerprints[split] = {'available_articles': len(pools[split]),
                              'selected_sha256': hash_json(chosen)}
    # Comprobar también igualdad exacta de las ventanas seleccionadas entre particiones.
    train_blocks = {tuple(x['context'] + x['target']) for x in selected['train']}
    if any(tuple(x['context'] + x['target']) in train_blocks for x in selected['validation']):
        raise RuntimeError('Hay bloques idénticos seleccionados entre train y validation; revisar corpus.')
    return selected, fingerprints


In [ ]:
@torch.inference_mode()
def human_perplexity(model, rows, cfg, device, pad_id):
    model.eval()
    nll, count = 0.0, 0
    for start in range(0, len(rows), cfg.micro_batch):
        batch = batch_tensors(rows[start:start+cfg.micro_batch], pad_id, device)
        output = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], use_cache=False)
        labels = batch['labels'][:, 1:].contiguous()
        loss = F.cross_entropy(output.logits[:, :-1, :].float().reshape(-1, output.logits.size(-1)),
                               labels.reshape(-1), reduction='sum', ignore_index=-100)
        nll += float(loss.cpu())
        count += int(labels.ne(-100).sum().item())
    assert count == len(rows) * cfg.target_tokens
    mean = nll / count
    if not math.isfinite(mean) or mean > 700:
        raise RuntimeError('Perplejidad no finita o fuera de rango.')
    return {'human_nll': mean, 'human_perplexity': math.exp(mean), 'ppl_target_tokens': count}


@torch.inference_mode()
def semantic_dispersion(texts, encoder, encoder_tokenizer, cfg, limit):
    # Mismas primeras W palabras que en diversidad léxica; sin descartar textos cortos.
    if any(len(words(t)) < limit for t in texts):
        return {'semantic_dispersion': None, 'embedding_truncated_texts': None}
    embeddings, truncated = [], 0
    encoder.eval()
    for start in range(0, len(texts), 16):
        batch = [' '.join(words(t)[:limit]) for t in texts[start:start+16]]
        lengths = [len(x) for x in encoder_tokenizer(batch, truncation=False)['input_ids']]
        truncated += sum(n > cfg.embedding_max_tokens for n in lengths)
        inputs = encoder_tokenizer(batch, padding=True, truncation=True,
                                   max_length=cfg.embedding_max_tokens, return_tensors='pt')
        output = encoder(**inputs).last_hidden_state
        mask = inputs['attention_mask'].unsqueeze(-1).to(output.dtype)
        pooled = (output * mask).sum(1) / mask.sum(1).clamp_min(1)
        embeddings.append(F.normalize(pooled, p=2, dim=1))
    matrix = torch.cat(embeddings)
    similarity = matrix @ matrix.T
    upper = torch.triu_indices(len(texts), len(texts), offset=1)
    dispersion = 1 - similarity[upper[0], upper[1]].mean().item()
    return {'semantic_dispersion': dispersion, 'embedding_truncated_texts': truncated}


def evaluate_stage(model, valid, tokenizer, encoder, enc_tokenizer, cfg,
                   device, forbidden, limit, folder, clock):
    with clock.measure('eval_generation', stage_id=folder.name):
        generated, seeds = sample_targets(model, valid, cfg, device, forbidden, 'validation_paired')
    texts = tokenizer.batch_decode(generated, skip_special_tokens=True)
    with clock.measure('eval_perplexity', stage_id=folder.name):
        metrics = human_perplexity(model, valid, cfg, device, tokenizer.pad_token_id)
    metrics.update(lexical_metrics(texts, limit))
    with clock.measure('eval_embeddings', stage_id=folder.name):
        metrics.update(semantic_dispersion(texts, encoder, enc_tokenizer, cfg, limit))
    with clock.measure('write_outputs', stage_id=folder.name):
        write_jsonl(folder/'validation_generations.jsonl', [
            {'id': row['id'], 'seed': seed, 'target_ids': ids, 'text': text}
            for row, seed, ids, text in zip(valid, seeds, generated, texts)])
        write_json(folder/'metrics.json', metrics)
    return metrics


In [ ]:
PilotConfig = Config


## 4. Configuración principal, checkpoints y persistencia

Revisiones del modelo, dataset y codificador fijadas a los commits del piloto. Un cambio
de parámetros, código o entorno esencial bloquea reanudación en el mismo directorio.

Se guardan checkpoints Safetensors para M0 y para continuar la cadena activa. Después
de confirmar el sucesor completo se purgan únicamente los pesos no iniciales anteriores
creados por este programa. Sus corpus, semillas, métricas y marcadores se conservan.
No se promete conservar los pesos de los 48 modelos. El máximo habitual ronda 2.5–3 GB
en pesos durante escritura, más datos; quedan transferencias acumuladas e I/O de Drive.

Los lotes de generación se guardan con hashes de entrada, productor y salida. Un ajuste
interrumpido antes de guardar su checkpoint se repite desde base/semilla; un ajuste ya
guardado continúa en evaluación. Los marcadores completos verifican archivos y evitan
reentrenar etapas confirmadas. No es una transacción distribuida ni garantiza sincronización
instantánea de Drive: un archivo incompleto o corrupto causa detención, no sustitución silenciosa.


In [ ]:
import csv
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import shutil
import sys
import time
from dataclasses import dataclass, asdict, replace
from datetime import datetime, timezone
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from safetensors.torch import save_model, load_model
import matplotlib.pyplot as plt


@dataclass(frozen=True)
class MainConfig(PilotConfig):
    train_examples: int = 512
    eval_examples: int = 128
    validation_examples: int = 128
    repetitions: int = 3
    save_checkpoints: bool = True
    cycles: int = 5
    alphas: tuple = (0.0, 0.5, 1.0)
    split_seed: int = 20261109  # Conserva las particiones del piloto.
    seed: int = 20261109
    model_revision: str = '02e3092372e8a8b20c64e29350a024a998a7c00d'
    dataset_revision: str = 'b04c8d1ceb2f5cd4588862100d08de323dccfbaa'
    embedding_revision: str = 'e8f8c211226b894fcb81acc59f3b34ba3efd5f42'


MAIN_CFG = MainConfig()


def read_record(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def environment_record():
    names = ('torch', 'transformers', 'datasets', 'safetensors', 'numpy', 'tokenizers', 'huggingface-hub')
    return {'python': platform.python_version(),
            'packages': {n: importlib.metadata.version(n) for n in names},
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
            'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version()}


def check_main_config(cfg, output_root):
    if sys.version_info < (3, 11) or tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 6):
        raise RuntimeError('Se requiere Python >=3.11 y PyTorch >=2.6.')
    if (cfg.language != 'es' or cfg.subset != '20231101.es' or cfg.train_examples != 512
            or cfg.eval_examples != 128 or cfg.validation_examples != 128
            or cfg.repetitions != 3 or cfg.cycles != 5 or cfg.alphas != (0.0, .5, 1.0)):
        raise ValueError('La configuración difiere del protocolo principal preparado.')
    if cfg.split_seed != 20261109:
        raise ValueError('No cambiar las particiones utilizadas en el piloto.')
    if not torch.cuda.is_available():
        raise RuntimeError('Se requiere GPU CUDA para esta configuración.')
    if cfg.effective_batch % cfg.micro_batch or cfg.train_examples % cfg.effective_batch:
        raise ValueError('Presupuesto y acumulación incompatibles.')
    # En Colab se exige persistencia en Drive: /content desaparece con la sesión.
    root = Path(output_root).resolve()
    if Path('/content').exists() and not root.is_relative_to(Path('/content/drive/MyDrive')):
        raise ValueError('Usar la carpeta persistente de MyDrive preparada en el notebook.')
    if Path('/content').exists() and not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Google Drive aún no está montado; completar esa celda manualmente.')


class MainClock:
    def __init__(self, root):
        self.root = root

    def measure(self, name, action):
        torch.cuda.synchronize()
        start = time.perf_counter()
        result = action()
        torch.cuda.synchronize()
        records = read_record(self.root/'timings.json') if (self.root/'timings.json').exists() else []
        records.append({'activity': name, 'seconds': time.perf_counter()-start,
                        'recorded_utc': datetime.now(timezone.utc).isoformat()})
        atomic_json(self.root/'timings.json', records)
        return result


### Preparación y reanudación


In [ ]:
def prepare_main_data(root, cfg, tokenizer, signature):
    folder = root/'data'
    if stage_complete(folder, signature):
        return read_record(folder/'corpus.json'), read_record(folder/'metadata.json')
    pools = {s: [] for s in ('train', 'validation', 'test')}
    counts = {s: 0 for s in pools}
    seen = set()
    stream = load_dataset(cfg.dataset, cfg.subset, split='train', streaming=True,
                           revision=cfg.dataset_revision, token=False)
    width = cfg.context_tokens + cfg.target_tokens
    for number, article in enumerate(stream):
        if number >= cfg.article_pool:
            break
        identity = str(article['id'])
        if identity in seen:
            raise ValueError('ID de artículo duplicado en el corpus.')
        seen.add(identity)
        split = article_partition(identity, cfg.split_seed)
        counts[split] += 1
        text = normalized_text(article['text'])
        ids = tokenizer(text, add_special_tokens=False, truncation=False, verbose=False)['input_ids']
        windows_count = len(ids)//width
        if not windows_count:
            continue
        offset = random.Random(stable_seed(cfg.split_seed, 'window', identity)).randrange(windows_count)*width
        block = ids[offset:offset+width]
        pools[split].append({'id': f'es:{split}:{identity}:{offset}', 'split': split,
            'article': identity, 'title': article.get('title', ''), 'url': article.get('url', ''),
            'text_sha256': hashlib.sha256(text.encode()).hexdigest(), 'offset': offset,
            'context': block[:cfg.context_tokens], 'target': block[cfg.context_tokens:]})
    selected = final_selection(pools, (cfg.train_examples, cfg.validation_examples, cfg.eval_examples), cfg.seed)
    # Se fija longitud exclusivamente con validación, nunca con resultados de prueba.
    validation = tokenizer.batch_decode([r['target'] for r in selected['validation']], skip_special_tokens=True)
    limit = min(cfg.max_words, min(len(words(t)) for t in validation))
    if limit < 4:
        raise ValueError('Validación no permite longitud léxica común suficiente.')
    metadata = {'word_limit': limit, 'scanned_articles': len(seen), 'split_counts': counts,
        'eligible_counts': {k: len(v) for k, v in pools.items()}, 'corpus_sha256': canonical_hash(selected),
        'split_seed': cfg.split_seed, 'test_used_for_tuning': False,
        'normalization': 'NFC; minúsculas solo en métricas; tildes y ñ conservadas'}
    atomic_json(folder/'corpus.json', selected)
    atomic_json(folder/'metadata.json', metadata)
    commit_stage(folder, signature, ['corpus.json', 'metadata.json'], {'kind': 'data_prepared'})
    return selected, metadata


def save_checkpoint(model, folder):
    pending, target = folder/'model.pending.safetensors', folder/'model.safetensors'
    save_model(model, str(pending))
    pending.replace(target)
    return file_hash(target)


def restore_checkpoint(model, folder, device='cuda'):
    record = read_record(folder/'fit.json')
    path = folder/'model.safetensors'
    if not path.is_file() or file_hash(path) != record['checkpoint_sha256']:
        raise ValueError(f'Checkpoint necesario ausente o modificado: {folder.name}')
    load_model(model, str(path), strict=True, device=device)
    if weight_hash(model) != record['final_weight_sha256']:
        raise ValueError('Los pesos restaurados no coinciden con la huella registrada.')
    model.eval()


def sampled_batches(model, rows, cfg, tokenizer, namespace, folder, signature, producer_hash):
    folder.mkdir(parents=True, exist_ok=True)
    generated = []
    for start in range(0, len(rows), cfg.sample_batch):
        chunk = rows[start:start+cfg.sample_batch]
        path = folder/f'batch_{start:05d}.json'
        input_hash = canonical_hash({'signature': signature, 'namespace': namespace, 'producer': producer_hash,
                                      'rows': chunk, 'seed': cfg.seed})
        outputs = cached_chunk(path, input_hash)
        if outputs is None:
            ids, seeds = sample_targets(model, chunk, cfg, 'cuda', tokenizer.all_special_ids,
                                         namespace, progress=False)
            outputs = [{'id': r['id'], 'seed': s, 'target_ids': t,
                        'text': tokenizer.decode(t, skip_special_tokens=True)}
                       for r, s, t in zip(chunk, seeds, ids)]
            save_chunk(path, input_hash, outputs)
        if len(outputs) != len(chunk) or [x['id'] for x in outputs] != [x['id'] for x in chunk]:
            raise ValueError('Lote guardado incompleto.')
        for row, output in zip(chunk, outputs):
            if (len(output['target_ids']) != cfg.target_tokens
                    or set(output['target_ids']) & set(tokenizer.all_special_ids)
                    or output['seed'] != stable_seed(cfg.seed, 'sampling', namespace, row['id'])):
                raise ValueError('Longitud o semilla de lote inválida.')
        generated.extend(outputs)
    return generated


def prune_previous_checkpoints(root, current, signature):
    # Solo pesos creados por este programa, de etapas completas no iniciales.
    # Se llama DESPUÉS de verificar y confirmar el sucesor completo.
    for job in make_plan():
        folder = root/'stages'/job['id']
        if folder == current or folder.name.endswith('_M0'):
            continue
        checkpoint = folder/'model.safetensors'
        if checkpoint.is_file() and stage_complete(folder, signature):
            checkpoint.unlink()


### Ejecución principal protegida


In [ ]:
def run_main(cfg, output_root, code_sha256, max_new_stages=None):
    check_main_config(cfg, output_root)
    if max_new_stages is not None and (not isinstance(max_new_stages, int) or max_new_stages < 1):
        raise ValueError('El límite de etapas debe ser positivo o None.')
    root = Path(output_root).resolve()
    torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    environment = environment_record()
    signature = freeze_protocol(root, {'config': asdict(cfg), 'code_sha256': code_sha256,
        'eos_policy': '128 tokens sin IDs especiales', 'word_pattern': WORD_PATTERN,
        'mixture_policy': 'subset sintético fijo por repetición/alpha; generar 512 antes de mezclar'}, environment)
    clock = MainClock(root)
    jobs = make_plan(cfg.repetitions, cfg.cycles, cfg.alphas)
    completed = [job for job in jobs if stage_complete(root/'stages'/job['id'], signature)]
    if len(completed) == len(jobs):
        atomic_json(root/'finished.json', {'status': 'complete', 'stages': len(jobs), 'signature': signature})
        print('Las 48 etapas ya están completas; no se vuelve a entrenar.')
        return root
    torch_smoke_checks()
    print('Entorno:', environment['gpu'], '| Etapas completas:', len(completed), '/', len(jobs), flush=True)
    tokenizer = AutoTokenizer.from_pretrained(cfg.model, revision=cfg.model_revision,
                                              token=False, trust_remote_code=False)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    data, metadata = clock.measure('prepare_data', lambda: prepare_main_data(root, cfg, tokenizer, signature))
    model = clock.measure('load_generator', lambda: AutoModelForCausalLM.from_pretrained(
        cfg.model, revision=cfg.model_revision, token=False, trust_remote_code=False,
        use_safetensors=False, weights_only=True, dtype=torch.float32,
        attn_implementation='eager').to('cuda'))
    model.config.pad_token_id = tokenizer.pad_token_id
    if not all(p.requires_grad for p in model.parameters()):
        raise RuntimeError('El ajuste debe actualizar todos los parámetros.')
    base, base_hash = cpu_state(model), weight_hash(model)
    model_record = root/'base_model.json'
    if model_record.exists() and read_record(model_record)['weight_hash'] != base_hash:
        raise ValueError('Los pesos preentrenados cambiaron respecto a la ejecución guardada.')
    atomic_json(model_record, {'weight_hash': base_hash, 'parameters': sum(p.numel() for p in model.parameters())})
    enc_tokenizer = AutoTokenizer.from_pretrained(cfg.embedding, revision=cfg.embedding_revision,
                                                 token=False, trust_remote_code=False)
    encoder = clock.measure('load_encoder', lambda: AutoModel.from_pretrained(cfg.embedding,
        revision=cfg.embedding_revision, token=False, trust_remote_code=False,
        use_safetensors=True, dtype=torch.float32).cpu().eval())
    new_count = 0
    for job in jobs:
        folder = root/'stages'/job['id']
        if stage_complete(folder, signature):
            continue
        folder.mkdir(parents=True, exist_ok=True)
        local_cfg = replace(cfg, seed=stable_seed(cfg.seed, 'replicate', job['rep']))
        print('Procesando:', job['id'], flush=True)
        if (folder/'fit.json').exists():
            restore_checkpoint(model, folder)  # Ajuste terminado; evaluación pendiente.
        else:
            if (folder/'training_corpus.json').exists() and (folder/'corpus_record.json').exists():
                corpus = read_record(folder/'training_corpus.json')
                corpus_record = read_record(folder/'corpus_record.json')
                if corpus_record != {'signature': signature, 'sha256': canonical_hash(corpus)}:
                    raise ValueError('Corpus parcial incompatible o modificado.')
            else:
                synthetic = None
                if job['alpha']:
                    parent = root/'stages'/job['parent']
                    if not stage_complete(parent, signature):
                        raise ValueError('El modelo padre no tiene una etapa completa.')
                    restore_checkpoint(model, parent)
                    outputs = clock.measure(job['id']+'/train_generation', lambda: sampled_batches(
                        model, data['train'], local_cfg, tokenizer,
                        f"train_a{job['alpha']}_g{job['cycle']}", folder/'train_batches', signature,
                        read_record(parent/'fit.json')['final_weight_sha256']))
                    synthetic = [row['target_ids'] for row in outputs]
                mixture_seed = stable_seed(cfg.seed, 'mixture', job['rep'], job['alpha'])
                corpus = mix_targets(data['train'], synthetic, job['alpha'] or 0.0, mixture_seed)
                atomic_json(folder/'training_corpus.json', corpus)
                atomic_json(folder/'corpus_record.json', {'signature': signature, 'sha256': canonical_hash(corpus)})
            if (len(corpus) != cfg.train_examples
                    or [x['id'] for x in corpus] != [x['id'] for x in data['train']]
                    or [x['context'] for x in corpus] != [x['context'] for x in data['train']]
                    or any(len(x['target']) != cfg.target_tokens for x in corpus)
                    or sum(x['synthetic'] for x in corpus) != round((job['alpha'] or 0)*cfg.train_examples)):
                raise ValueError('El corpus guardado no respeta el presupuesto o la mezcla.')
            if any(not row['synthetic'] and row['target'] != original['target']
                   for row, original in zip(corpus, data['train'])):
                raise ValueError('Objetivo original cambiado en el corpus guardado.')
            fit_seed = stable_seed(cfg.seed, 'fit', job['rep'], job['cycle'])
            fit = clock.measure(job['id']+'/fit', lambda: fit_one(model, base, base_hash,
                corpus, local_cfg, fit_seed, 'cuda', tokenizer.pad_token_id))
            fit['corpus_sha256'] = canonical_hash(corpus)
            fit['checkpoint_sha256'] = clock.measure(job['id']+'/checkpoint', lambda: save_checkpoint(model, folder))
            atomic_json(folder/'fit.json', fit)
        fit = read_record(folder/'fit.json')
        if fit['corpus_sha256'] != canonical_hash(read_record(folder/'training_corpus.json')):
            raise ValueError('El corpus del ajuste guardado cambió.')
        samples = clock.measure(job['id']+'/test_generation', lambda: sampled_batches(model,
            data['test'], local_cfg, tokenizer, 'test_paired', folder/'test_batches', signature,
            fit['final_weight_sha256']))
        texts = [row['text'] for row in samples]
        metrics = clock.measure(job['id']+'/test_ppl', lambda: human_perplexity(model,
            data['test'], local_cfg, 'cuda', tokenizer.pad_token_id))
        metrics.update(lexical_metrics(texts, metadata['word_limit']))
        metrics.update(clock.measure(job['id']+'/embeddings', lambda: semantic_dispersion(
            texts, encoder, enc_tokenizer, local_cfg, metadata['word_limit'])))
        metrics.update(stage=job['id'], rep=job['rep'], cycle=job['cycle'], alpha=job['alpha'])
        atomic_json(folder/'test_generations.json', samples)
        atomic_json(folder/'metrics.json', metrics)
        commit_stage(folder, signature, ['training_corpus.json', 'corpus_record.json', 'fit.json', 'test_generations.json', 'metrics.json'], job)
        assert stage_complete(folder, signature)
        prune_previous_checkpoints(root, folder, signature)
        new_count += 1
        print('Etapa confirmada:', job['id'], '(métricas guardadas; resumen al finalizar)', flush=True)
        if max_new_stages is not None and new_count >= max_new_stages:
            print('Límite voluntario de etapas alcanzado. Reanudar con la misma carpeta/configuración.')
            return root
    atomic_json(root/'finished.json', {'status': 'complete', 'stages': len(jobs),
        'signature': signature, 'finished_utc': datetime.now(timezone.utc).isoformat()})
    print('Estudio principal completo: 48 etapas. Puede ejecutar el resumen.')
    return root


### Resumen del estudio principal


In [ ]:
def summarize_main(root):
    root = Path(root)
    protocol = read_record(root/'protocol.json')
    signature = protocol['signature']
    plan = make_plan()
    if not all(stage_complete(root/'stages'/job['id'], signature) for job in plan):
        raise RuntimeError('El resumen de prueba se muestra al completar las 48 etapas.')
    rows = [read_record(root/'stages'/job['id']/'metrics.json') for job in plan]
    with (root/'results.csv').open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    atomic_json(root/'results.json', rows)
    keys = ['distinct_2', 'distinct_3', 'repeat_4_mean', 'semantic_dispersion', 'human_perplexity']
    contrasts = []
    for row in rows:
        if row['alpha'] not in (.5, 1.0):
            continue
        initial = next(r for r in rows if r['rep'] == row['rep'] and r['cycle'] == 0)
        control = next(r for r in rows if r['rep'] == row['rep'] and r['cycle'] == row['cycle'] and r['alpha'] == 0)
        contrast = {k: row[k] for k in ('rep', 'cycle', 'alpha')}
        for metric in keys:
            contrast[metric+'_vs_M0'] = None if row[metric] is None or initial[metric] is None else row[metric]-initial[metric]
            contrast[metric+'_vs_control'] = None if row[metric] is None or control[metric] is None else row[metric]-control[metric]
        contrasts.append(contrast)
    atomic_json(root/'contrasts.json', contrasts)
    with (root/'contrasts.csv').open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(contrasts[0]))
        writer.writeheader(); writer.writerows(contrasts)
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, metric in zip(axes.flat, keys):
        for alpha in (0.0, .5, 1.0):
            means, lows, highs = [], [], []
            for cycle in range(6):
                values = [r[metric] for r in rows if r['cycle'] == cycle
                          and (cycle == 0 or r['alpha'] == alpha)]
                if any(v is None for v in values):
                    means.append(float('nan')); lows.append(float('nan')); highs.append(float('nan'))
                else:
                    means.append(sum(values)/len(values)); lows.append(min(values)); highs.append(max(values))
            ax.plot(range(6), means, marker='o', label=f'{alpha:.0%}')
            ax.fill_between(range(6), lows, highs, alpha=.12)
        ax.set(title=metric, xlabel='Ciclo'); ax.grid(alpha=.3)
    axes.flat[0].legend(title='Objetivos sintéticos')
    axes.flat[-1].axis('off')
    fig.suptitle('Estudio principal: media y rango de tres cadenas; sin p-valores')
    fig.tight_layout()
    fig.savefig(root/'results.png', dpi=160, bbox_inches='tight')
    plt.show()
    # Exportación ligera: excluye checkpoints, lotes parciales y directorios ajenos.
    import zipfile
    archive = root/'resultados_sin_pesos.zip'
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as target:
        for name in ('protocol.json', 'base_model.json', 'finished.json', 'timings.json',
                     'results.csv', 'results.json', 'results.png', 'contrasts.csv', 'contrasts.json'):
            if (root/name).exists():
                target.write(root/name, name)
        for folder in [root/'data']+[root/'stages'/job['id'] for job in plan]:
            for path in folder.glob('*.json'):
                target.write(path, path.relative_to(root))
    print('Resumen y exportación sin pesos:', archive)
    return archive


## 5. Comprobar checkpoints con un modelo diminuto

Este control adicional está preparado para ejecutarse en Colab. Guarda un GPT-2 diminuto,
modifica un parámetro y restaura los pesos verificando la misma huella. No descarga el
generador ni utiliza el corpus de prueba. La comprobación aún no se ha ejecutado durante
la preparación local de este notebook.


In [ ]:
import tempfile
torch_smoke_checks()
with tempfile.TemporaryDirectory(dir=Path.cwd()) as checkpoint_temp:
    checkpoint_folder = Path(checkpoint_temp)
    tiny = GPT2LMHeadModel(GPT2Config(vocab_size=32, n_embd=16, n_inner=32,
        n_layer=1, n_head=2, n_positions=32, bos_token_id=2, eos_token_id=2, pad_token_id=1)).cpu()
    signature_before = weight_hash(tiny)
    disk_hash = save_checkpoint(tiny, checkpoint_folder)
    atomic_json(checkpoint_folder/'fit.json', {'checkpoint_sha256': disk_hash,
                                              'final_weight_sha256': signature_before})
    with torch.no_grad():
        next(tiny.parameters()).add_(.01)
    assert weight_hash(tiny) != signature_before
    restore_checkpoint(tiny, checkpoint_folder, device='cpu')
    assert weight_hash(tiny) == signature_before
    del tiny
print('Checkpoint diminuto guardado y restaurado correctamente.')


## 6. Ruta persistente y autorización de Drive

La carpeta del experimento se crea al iniciar, no al importar. Mantener una sola sesión
escribiendo en ella. Elegir una ruta nueva si ya tiene archivos ajenos o un protocolo diferente.

`PREPARE_DRIVE=True` permite solicitar el montaje; el usuario debe completar la autorización
de su cuenta. El programa no introduce credenciales ni aprueba ese acceso por su cuenta.
No cambiar a `/content` para una ejecución que se pretenda reanudar tras perder la VM.


In [ ]:
SOURCE_SHA256 = '42531e6eece94da4767215350947c4d321ca29ddf29cf403c6ef545b592c5624'
OUTPUT_ROOT = Path('/content/drive/MyDrive/IMAC/estudio_principal_es_v1')
PREPARE_DRIVE = False
RUN_EXPERIMENT = False
MAX_NEW_STAGES = 1  # Primera comprobación de I/O; None continúa todas las etapas restantes.

if PREPARE_DRIVE and not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Montaje no solicitado o Drive ya disponible. Ruta:', OUTPUT_ROOT)
print('Diseño:', len(make_plan()), 'ajustes; ejecución activada:', RUN_EXPERIMENT)


## 7. Ejecutar o reanudar

Al activar la bandera, el mismo procedimiento detecta etapas completas y continúa las
pendientes. Los 48 ajustes no se iniciaron durante la creación de este archivo. Primero
se recomienda confirmar una etapa completa y probar la reanudación con su checkpoint.

La muestra sintética de 50 % usa los mismos 256 IDs durante los ciclos de una repetición.
Se generan 512 continuaciones en cada ciclo no humano y se aplica esa selección fija.
Los 128 contextos de prueba son comunes a condiciones/repeticiones. Sus métricas se guardan
por etapa, pero el programa muestra el resumen conjunto al terminar, sin utilizarlas para
decidir parámetros, checkpoints o detención temprana.


In [ ]:
if RUN_EXPERIMENT:
    MAIN_RUN = run_main(MAIN_CFG, OUTPUT_ROOT, SOURCE_SHA256, max_new_stages=MAX_NEW_STAGES)
else:
    print('Configuración preparada. No se han iniciado ajustes principales.')


## 8. Resumen final y exportación

Las gráficas presentan media y rango de tres cadenas, no 128 réplicas de entrenamiento.
Se exportan resultados y contrastes por repetición frente a M0 y frente al control del
mismo ciclo. No se calculan p-valores con estos tamaños pequeños.
El ZIP excluye pesos, pero Drive conserva los checkpoints necesarios para reanudar.


In [ ]:
if RUN_EXPERIMENT and (OUTPUT_ROOT/'finished.json').exists():
    MAIN_ARCHIVE = summarize_main(OUTPUT_ROOT)
    from google.colab import files
    files.download(str(MAIN_ARCHIVE))
else:
    print('Resumen disponible al confirmar las 48 etapas; no se muestran métricas parciales de prueba.')


## Fuentes y límites

- [Generador español](https://huggingface.co/datificate/gpt2-small-spanish),
  [Wikipedia](https://huggingface.co/datasets/wikimedia/wikipedia) y
  [MiniLM multilingüe](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2).
- [Safetensors: save_model/load_model y pesos compartidos](https://huggingface.co/docs/safetensors/main/en/api/torch).
- [Colab: límites de recursos, sesiones y Drive](https://research.google.com/colaboratory/faq.html).

Consultadas el 9 de noviembre de 2026. Conservar licencias/atribución de recursos al
redistribuir. Semillas e igualdad de versiones facilitan repetir, sin garantizar identidad
entre entornos. El piloto no probó la escritura de checkpoints en Drive ni la duración
de 48 ajustes. Se verificaron localmente calendario, particiones, corrupción y reanudación
de registros/lotes; siguen pendientes la prueba integrada y la ejecución principal.
